In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [171]:
import numpy as np
import pandas as pd
import os
import seaborn as sns
from tqdm import tqdm
import matplotlib.pyplot as plt
import xgboost, lightgbm, catboost

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import config as cnf 
import utils as utl

### Вывод

In [143]:
df_train = pd.read_csv(cnf.path['train_path'])

# Обработка данных
df_train_priz = df_train[['Survived', 'Pclass', 'Age', 'Sex',  'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
df_train_priz['Age'] = df_train_priz['Age'].fillna(df_train_priz['Age'].median())
df_train_priz['Embarked'] = df_train_priz['Embarked'].fillna('S')

for col in ['Sex', 'Embarked']:
    df_train_priz[col] = df_train_priz[col].astype('category')

X = df_train_priz.drop('Survived', axis=1)
# Второй датасет с OneHotCoding (Не для бустингов)
X_ohe = pd.get_dummies(X, dtype=float, drop_first=True)

y = df_train_priz['Survived']

# Подготовка параметров
combo_param = utl.parametrs(cnf.RandomForestParam)
combo_param += utl.parametrs(cnf.LogRegParam)
combo_param += utl.parametrs(cnf.KNNParam)
combo_param += utl.parametrs(cnf.DecisionTreeParam)
combo_param += utl.parametrs(cnf.CatBoostParam)
combo_param += utl.parametrs(cnf.LGBMParam)
combo_param += utl.parametrs(cnf.XGBoostParam)


# Подготовка кросс валидации

results = []
SK_Fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=cnf.random_state)

for combo in tqdm(combo_param):
    
    params = combo.copy()
    name_model = params.pop('name_model')
    use_scaler = params.pop('use_scaler')
    use_category = params.pop('use_category')
    
    # Флаг на категориальные признаки - если False, делаем OneHotCoding
    X_cur = X if use_category else X_ohe
    
    array_train_accuracy = []
    array_test_accuracy = []
    
    model = cnf.models[name_model](**params)
    
    for i, (train_index, test_index) in enumerate(SK_Fold.split(X, y)):
        
        current_X_train = X_cur.iloc[train_index]
        current_y_train = y.iloc[train_index]
        current_X_test  = X_cur.iloc[test_index]
        current_y_test  = y.iloc[test_index]

        # scaler - если True, делаем StandardScaler
        if use_scaler:
            scaler = StandardScaler()
            scaler.fit(current_X_train)
            current_X_train = scaler.transform(current_X_train)
            current_X_test = scaler.transform(current_X_test)
            
        model_clone = clone(model)
        model_clone.fit(current_X_train, current_y_train)
        
        train_accuracy = model_clone.score(current_X_train, current_y_train)
        test_accuracy = model_clone.score(current_X_test, current_y_test)

        array_train_accuracy.append(train_accuracy)
        array_test_accuracy.append(test_accuracy)

    mean_train_accuracy = float(np.mean(array_train_accuracy))
    mean_train_std = float(np.std(array_train_accuracy))
    mean_test_accuracy = float(np.mean(array_test_accuracy))
    mean_test_std = float(np.std(array_test_accuracy))
    
    # Переименовываем параметры в Катбусте
    combo_row = {cnf.rename_params.get(k, k): v for k, v in combo.items()}
    
    results.append({
        **combo_row
        , 'train_accuracy'  :   mean_train_accuracy
        , 'train_std'       :   mean_train_std
        , 'test_accuracy'   :   mean_test_accuracy
        , 'test_std'        :   mean_test_std
    })

100%|██████████| 10/10 [01:32<00:00,  9.28s/it]


In [144]:
results = pd.DataFrame(results)
results['gap'] = results['train_accuracy'] - results['test_accuracy']

# Порядок колонок: сначала модель, потом метрики, потом гиперпараметры
metrics = ['test_accuracy', 'test_std', 'train_accuracy', 'train_std', 'gap']
service = ['name_model', 'use_scaler', 'random_state', 'verbose', 'allow_writing_files', 'cat_features', 'enable_categorical']
param_cols = [c for c in results.columns if c not in metrics + service]

results = results[['name_model'] + metrics + param_cols]

# Сортировка: лучшие по test_accuracy сверху
results = results.sort_values('test_accuracy', ascending=False).reset_index(drop=True)
results.head(50)

,name_model,test_accuracy,test_std,train_accuracy,train_std,gap,max_depth,max_features,min_samples_leaf,n_estimators,...,max_iter,n_neighbors,p,weights,criterion,reg_lambda,learning_rate,num_leaves,min_child_weight,subsample
0,XGBoost,0.848478,0.022515,0.890011,0.003411,0.041533,4.0,NaN,NaN,500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.03,NaN,5.0,0.8
1,RandomForest,0.843971,0.015454,0.912457,0.003406,0.068486,10.0,0.5,2.0,200.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DecisionTree,0.842866,0.013370,0.874576,0.007357,0.031710,8.0,NaN,5.0,NaN,...,NaN,NaN,NaN,NaN,gini,NaN,NaN,NaN,NaN,NaN
3,LGBM,0.841730,0.020977,0.896743,0.006650,0.055013,NaN,NaN,20.0,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.10,8.0,NaN,NaN
4,CatBoost,0.836131,0.024254,0.939955,0.003908,0.103824,6.0,NaN,NaN,300.0,...,NaN,NaN,NaN,NaN,NaN,3.0,0.10,NaN,NaN,NaN
5,KNN,0.820444,0.013956,0.842876,0.008031,0.022431,NaN,NaN,NaN,NaN,...,NaN,11.0,1.0,uniform,NaN,NaN,NaN,NaN,NaN,NaN
6,KNN,0.815963,0.016872,0.847643,0.004494,0.031680,NaN,NaN,NaN,NaN,...,NaN,9.0,1.0,uniform,NaN,NaN,NaN,NaN,NaN,NaN
7,LogReg,0.803584,0.016743,0.810325,0.003590,0.006741,NaN,NaN,NaN,NaN,...,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,LogReg,0.796874,0.017698,0.805836,0.002268,0.008962,NaN,NaN,NaN,NaN,...,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,LogReg,0.748603,0.018159,0.754769,0.004670,0.006166,NaN,NaN,NaN,NaN,...,1000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [141]:
results_500 = results

In [ ]:
import importlib
importlib.reload(utl)          # перезагружаем utils без перезапуска ядра

In [172]:
save_path = utl.next_result_path(cnf.path['file_path'], cnf.exp_name)
results.to_excel(save_path, index=False)
print(save_path)

C:\Users\Vsani\ML\Titanic\results\ml_resuts7_ААА.xlsx
